## Nettoyage & Ingénierie des caractéristiques 

###  I- Import des bibliothèques necessaires 

In [27]:
import re
import string
import numpy  as np
import pandas as pd
from pathlib    import Path
from datetime   import datetime
from urllib.parse import urlparse, parse_qs
from scipy.stats  import entropy as scipy_entropy
from sklearn.preprocessing import StandardScaler

print('✅ Imports OK')

✅ Imports OK


### II-Chemins vers le fichiers 

In [28]:


INPUT_PATH  = r"C:\Users\DELL\Desktop\projet\logs_output\Ancien_site web\Access_log\Combined_All_Logs.csv"
OUTPUT_PATH = r"C:\Users\DELL\Desktop\projet\logs_output\Ancien_site web\Access_log\Access_log_cleaned.csv"

print(f'Entrée : {INPUT_PATH}')
print(f'Sortie : {OUTPUT_PATH}')

Entrée : C:\Users\DELL\Desktop\projet\logs_output\Ancien_site web\Access_log\Combined_All_Logs.csv
Sortie : C:\Users\DELL\Desktop\projet\logs_output\Ancien_site web\Access_log\Access_log_cleaned.csv


### 3- Chargement et inspection initiale 

In [29]:


# Chargement : tout en string pour éviter les conversions automatiques
df = pd.read_csv(INPUT_PATH, dtype=str, low_memory=False)

print('=' * 65)
print('— INSPECTION INITIALE')
print('=' * 65)
print(f'  Lignes    : {len(df):,d}')
print(f'  Colonnes  : {len(df.columns)}')

print(f'\n  Colonnes disponibles :')
for col in df.columns:
    print(f'    - {col}')

print(f'\n  Distribution log_type :')
print(df['log_type'].value_counts().to_string())

# ── Tableau : valeurs uniques + manquantes ──────────────
print(f'\n  Analyse des valeurs uniques et manquantes :')
print(f'  {"Variable":<25} {"Uniques":>10} {"Manquantes":>12} {"Type":<10}')
print(f'  {"-"*25} {"-"*10} {"-"*12} {"-"*10}')
for col in df.columns:
    n_unique  = df[col].nunique()
    n_missing = df[col].isnull().sum()
    dtype     = df[col].dtype
    print(f'  {col:<25} {n_unique:>10,d} {n_missing:>12,d} {str(dtype):<10}')

— INSPECTION INITIALE
  Lignes    : 42,899
  Colonnes  : 12

  Colonnes disponibles :
    - ip
    - date
    - method
    - url
    - protocol
    - status_code
    - response_size
    - referer
    - user_agent
    - source_file
    - source_path
    - log_type

  Distribution log_type :
log_type
apache_access    42899

  Analyse des valeurs uniques et manquantes :
  Variable                     Uniques   Manquantes Type      
  ------------------------- ---------- ------------ ----------
  ip                             5,731            0 str       
  date                          38,619            0 str       
  method                             3            0 str       
  url                           10,310            0 str       
  protocol                           2            0 str       
  status_code                        2            0 str       
  response_size                    389            0 str       
  referer                          122            0 str       


### 4 —  VALEURS MANQUANTES
### Stratégie du mémoire : fillna('unknown') pour préserver

In [30]:


df_clean = df.copy()

# Masques par type de log
is_access = df_clean['log_type'] == 'apache_access'
is_error  = df_clean['log_type'] == 'apache_error'
is_syslog = df_clean['log_type'] == 'syslog'

# ── TOUTES COLONNES TEXTE : NaN → 'unknown' ──────────────────
# Mémoire : "remplace les valeurs manquantes par une
# catégorie 'unknown' pour éviter de supprimer ces lignes
# potentiellement informatives"
cols_texte = ['ip', 'method', 'url', 'protocol', 'referer',
              'user_agent', 'hostname', 'service', 'message',
              'error_code', 'module', 'level']
for col in cols_texte:
    if col in df_clean.columns:
        # Remplacer aussi le tiret '-' (valeur nulle Apache)
        df_clean[col] = df_clean[col].replace('-', np.nan).fillna('unknown')

# ── response_size : NaN ou '-' → 0 ──────────────────────────
if 'response_size' in df_clean.columns:
    df_clean['response_size'] = (
        df_clean['response_size']
        .replace('-', np.nan)
        .fillna('0')
    )

# ── pid / tid : NaN → -1 (absence de processus) ─────────────
for col in ['pid', 'tid']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('-1')

# ── status_code : NaN → 0 (inconnu) ─────────────────────────
if 'status_code' in df_clean.columns:
    df_clean['status_code'] = df_clean['status_code'].fillna('0')

# ── Rapport ──────────────────────────────────────────────────
print('=' * 65)
print(' — VALEURS MANQUANTES TRAITÉES')
print('=' * 65)
print(f'  Lignes avant : {len(df):,d}')
print(f'  Lignes après : {len(df_clean):,d}  (aucune ligne supprimée)')
print(f'  NaN restants : {df_clean.isnull().sum().sum():,d}')

§3.3.9 — VALEURS MANQUANTES TRAITÉES
  Lignes avant : 42,899
  Lignes après : 42,899  (aucune ligne supprimée)
  NaN restants : 0


### 5 Normalisation des dates 

In [31]:


def parse_date(value: str, log_type: str) -> pd.Timestamp:
    """Convertit une chaîne de date selon le log_type → pd.Timestamp UTC."""
    if not value or str(value).strip() in ('', 'unknown', 'nan'):
        return pd.NaT
    value = str(value).strip()
    try:
        if log_type == 'apache_access':
            # '15/Aug/2021:03:38:42 +0000'
            return pd.to_datetime(value, format='%d/%b/%Y:%H:%M:%S %z', utc=True)
        elif log_type == 'apache_error':
            # 'Sun Aug 15 03:34:01.562068 2021'
            return pd.to_datetime(value, format='%a %b %d %H:%M:%S.%f %Y', utc=True)
        elif log_type == 'syslog':
            # 'Aug 15 03:34:01'  → ajouter l'année courante
            year = datetime.now().year
            return pd.to_datetime(f'{year} {value}', format='%Y %b %d %H:%M:%S', utc=True)
    except Exception:
        pass
    try:
        return pd.to_datetime(value, utc=True)
    except Exception:
        return pd.NaT

# Appliquer
df_clean['datetime'] = df_clean.apply(
    lambda row: parse_date(row.get('date', ''), row.get('log_type', '')), axis=1
)

# Supprimer les lignes sans datetime valide (données non parsables)
avant = len(df_clean)
df_clean = df_clean[df_clean['datetime'].notna()].copy()

# Supprimer l'ancienne colonne 'date' brute
df_clean = df_clean.drop(columns=['date'], errors='ignore')

# datetime en première colonne + tri chronologique
cols = ['datetime'] + [c for c in df_clean.columns if c != 'datetime']
df_clean = df_clean[cols].sort_values('datetime').reset_index(drop=True)

print('=' * 65)
print('— DATES NORMALISÉES (ISO 8601 / UTC)')
print('=' * 65)
print(f'  Lignes supprimées (datetime invalide) : {avant - len(df_clean):,d}')
print(f'  Période couverte :')
print(f'    Début : {df_clean["datetime"].min()}')
print(f'    Fin   : {df_clean["datetime"].max()}')

§3.3.3 — DATES NORMALISÉES (ISO 8601 / UTC)
  Lignes supprimées (datetime invalide) : 0
  Période couverte :
    Début : 2021-07-18 03:28:12+00:00
    Fin   : 2021-08-19 08:25:53+00:00


### 6 Suppressions des doublons 

In [32]:


avant = len(df_clean)

# Clés de déduplication par log_type
cols_communes = ['datetime', 'log_type', 'source_file']

df_access = df_clean[df_clean['log_type'] == 'apache_access'].copy()
df_error  = df_clean[df_clean['log_type'] == 'apache_error'].copy()
df_syslog = df_clean[df_clean['log_type'] == 'syslog'].copy()

df_access = df_access.drop_duplicates(
    subset=[c for c in cols_communes + ['ip', 'method', 'url', 'status_code']
            if c in df_access.columns], keep='first')

df_error = df_error.drop_duplicates(
    subset=[c for c in cols_communes + ['message']
            if c in df_error.columns], keep='first')

df_syslog = df_syslog.drop_duplicates(
    subset=[c for c in cols_communes + ['hostname', 'service', 'message']
            if c in df_syslog.columns], keep='first')

df_clean = pd.concat([df_access, df_error, df_syslog], ignore_index=True)
df_clean = df_clean.sort_values('datetime').reset_index(drop=True)

print('=' * 65)
print('— DOUBLONS SUPPRIMÉS')
print('=' * 65)
print(f'  Avant : {avant:,d}  |  Après : {len(df_clean):,d}  |  Supprimés : {avant - len(df_clean):,d}')

— DOUBLONS SUPPRIMÉS
  Avant : 42,899  |  Après : 41,433  |  Supprimés : 1,466


### 7 Distribution des variables 

In [33]:


print('=' * 65)
print('— DISTRIBUTION DES VARIABLES')
print('=' * 65)

# Distribution de 'method' ()
if 'method' in df_clean.columns:
    print('\n  Méthodes HTTP les plus fréquentes :')
    counts = df_clean['method'].value_counts()
    total  = len(df_clean)
    for val, cnt in counts.head(10).items():
        print(f'    {val:<15} {cnt:>8,d}  ({cnt/total*100:5.1f}%)')

# Distribution de 'status_code' ()
if 'status_code' in df_clean.columns:
    print('\n  Distribution des codes de statut HTTP :')
    counts = df_clean['status_code'].value_counts().sort_index()
    for val, cnt in counts.items():
        if cnt > 0:
            print(f'    {val:<8} {cnt:>8,d}  ({cnt/total*100:5.1f}%)')

# Distribution de 'referer' (— top 10)
if 'referer' in df_clean.columns:
    print('\n  Top 10 des référents les plus fréquents :')
    counts = df_clean[df_clean['referer'] != 'unknown']['referer'].value_counts()
    for val, cnt in counts.head(10).items():
        print(f'    {str(val)[:50]:<52} {cnt:>8,d}  ({cnt/total*100:5.1f}%)')

# Distribution de 'user_agent' (— top 10)
if 'user_agent' in df_clean.columns:
    print('\n  Top 10 des User-Agents les plus fréquents :')
    counts = df_clean[df_clean['user_agent'] != 'unknown']['user_agent'].value_counts()
    for val, cnt in counts.head(10).items():
        print(f'    {str(val)[:50]:<52} {cnt:>8,d}  ({cnt/total*100:5.1f}%)')

— DISTRIBUTION DES VARIABLES

  Méthodes HTTP les plus fréquentes :
    GET               36,898  ( 89.1%)
    HEAD               4,374  ( 10.6%)
    POST                 161  (  0.4%)

  Distribution des codes de statut HTTP :
    301            22  (  0.1%)
    302        41,411  ( 99.9%)

  Top 10 des référents les plus fréquents :
    https://www.google.com/                                 2,360  (  5.7%)
    http://www.google.com/                                    567  (  1.4%)
    https://www.google.com/search                              79  (  0.2%)
    cic.univ-lome.tg                                           72  (  0.2%)
    gbif.univ-lome.tg                                          44  (  0.1%)
    android-app://com.google.android.googlequicksearch         41  (  0.1%)
    http://hotspot.technoplus.net/                             30  (  0.1%)
    binance.com                                                30  (  0.1%)
    android-app://com.google.android.googlequicksearch 

### 8- Feature Engineering

In [34]:

# ── Helpers ──────────────────────────────────────────────────
# Definition des caracteres spéciaux 
SPECIAL_CHARS = set(string.punctuation)

# Fonction de décompte des caractères spéciaux

def count_special(text: str) -> int:
    """Nombre de caractères spéciaux dans une chaîne."""
    if not text or text == 'unknown':
        return 0
    return sum(1 for c in str(text) if c in SPECIAL_CHARS)


#	Fonction d'entropie de Shannon pour une chaîne de caractères

def shannon_entropy(text: str) -> float:
    """Entropie de Shannon des caractères d'une chaîne (§3.4.6)."""
    if not text or text == 'unknown':
        return 0.0
    chars = list(str(text))
    _, counts = np.unique(chars, return_counts=True)
    probs = counts / counts.sum()
    return float(scipy_entropy(probs, base=2))

#	Entropie de Shannon des segments du chemin URL 

def path_entropy(url: str) -> float:
    """Entropie de Shannon des segments du chemin URL (§3.4.5)."""
    if not url or url == 'unknown':
        return 0.0
    try:
        path = urlparse(str(url)).path
        segments = [s for s in path.split('/') if s]
        if not segments:
            return 0.0
        entropies = [shannon_entropy(s) for s in segments]
        return float(np.sum(entropies))
    except Exception:
        return 0.0

# ────────────────────────────────────────────────────────────
# A. VARIABLES TEMPORELLES  
# ────────────────────────────────────────────────────────────
print('  [1/6] Variables temporelles...')

# Encodage cyclique de l'heure (sin/cos)
hour = df_clean['datetime'].dt.hour
df_clean['date_hour_sin'] = np.sin(2 * np.pi * hour / 24)
df_clean['date_hour_cos'] = np.cos(2 * np.pi * hour / 24)

# Temps écoulé en secondes depuis la première requête
t_min = df_clean['datetime'].min()
df_clean['date_normalized'] = (
    df_clean['datetime'] - t_min
).dt.total_seconds()

# ────────────────────────────────────────────────────────────
# B. VARIABLES LIÉES À L'IP  
# ────────────────────────────────────────────────────────────
print('  [2/6] Variables IP...')

# Nombre total de requêtes par IP
ip_counts = df_clean.groupby('ip')['datetime'].transform('count')
df_clean['ip_request_count'] = ip_counts

# Durée (secondes) entre première et dernière requête par IP
ip_min = df_clean.groupby('ip')['datetime'].transform('min')
ip_max = df_clean.groupby('ip')['datetime'].transform('max')
df_clean['ip_time_spread'] = (ip_max - ip_min).dt.total_seconds().fillna(0)

# Taux de requêtes par seconde
spread_safe = df_clean['ip_time_spread'].replace(0, 1)
df_clean['ip_request_rate'] = df_clean['ip_request_count'] / spread_safe

# Nombre d'IPs uniques dans le sous-réseau /24
df_clean['_subnet'] = df_clean['ip'].apply(
    lambda x: '.'.join(str(x).split('.')[:3]) if str(x).count('.') >= 2 else x
)
subnet_counts = df_clean.groupby('_subnet')['ip'].transform('nunique')
df_clean['ip_subnet_activity'] = subnet_counts
df_clean = df_clean.drop(columns=['_subnet'])

# Taux d'erreurs par IP (status >= 400)
is_numeric_status = pd.to_numeric(df_clean['status_code'], errors='coerce').fillna(0)
df_clean['_is_error'] = (is_numeric_status >= 400).astype(int)
df_clean['ip_error_rate'] = df_clean.groupby('ip')['_is_error'].transform('mean')
df_clean = df_clean.drop(columns=['_is_error'])

# Écart-type des intervalles entre requêtes consécutives par IP
df_sorted = df_clean.sort_values(['ip', 'datetime'])
intervals  = df_sorted.groupby('ip')['datetime'].diff().dt.total_seconds().fillna(0)
ip_std_map = intervals.groupby(df_sorted['ip']).std().fillna(0)
df_clean['ip_interval_std'] = df_clean['ip'].map(ip_std_map)

# Nombre de méthodes HTTP distinctes par IP
if 'method' in df_clean.columns:
    method_div = df_clean.groupby('ip')['method'].transform('nunique')
    df_clean['ip_method_diversity'] = method_div

# ────────────────────────────────────────────────────────────
# C. VARIABLES LIÉES AU STATUT  
# ────────────────────────────────────────────────────────────
print('  [3/6] Variables statut...')

def categorize_status(val):
    try:
        s = int(float(val))
        if 200 <= s < 300: return '2xx'
        if 300 <= s < 400: return '3xx'
        if 400 <= s < 500: return '4xx'
        if 500 <= s < 600: return '5xx'
        return 'other'
    except Exception:
        return 'other'

if 'status_code' in df_clean.columns:
    df_clean['status_code_category'] = df_clean['status_code'].apply(categorize_status)

# ────────────────────────────────────────────────────────────
# D. VARIABLES LIÉES À L'URL  
# ────────────────────────────────────────────────────────────
print('  [4/6] Variables URL...')

url_col = 'url' if 'url' in df_clean.columns else None

if url_col:
    parsed_urls = df_clean[url_col].apply(
        lambda x: urlparse(str(x)) if x and x != 'unknown' else None
    )

    # Chemin hiérarchique (plusieurs segments /)
    df_clean['is_hierarchical'] = df_clean[url_col].apply(
        lambda x: 1 if str(x).count('/') > 1 else 0
    )

    # Profondeur du chemin
    df_clean['url_path_depth'] = parsed_urls.apply(
        lambda p: len([s for s in p.path.split('/') if s]) if p else 0
    )

    # Entropie du chemin
    df_clean['url_path_struct_entropy'] = df_clean[url_col].apply(path_entropy)

    # Nombre de paramètres query
    df_clean['nbre_total_url_query_params'] = parsed_urls.apply(
        lambda p: len(parse_qs(p.query)) if p else 0
    )

    # Caractères spéciaux dans query
    df_clean['url_query_special_chars_count'] = parsed_urls.apply(
        lambda p: count_special(p.query) if p else 0
    )

    # Caractères spéciaux dans le chemin
    df_clean['url_path_special_chars_count'] = parsed_urls.apply(
        lambda p: count_special(p.path) if p else 0
    )

    # Fragment
    df_clean['fragment_param_count'] = parsed_urls.apply(
        lambda p: len(parse_qs(p.fragment)) if p else 0
    )
    df_clean['special_char_fragment_count'] = parsed_urls.apply(
        lambda p: count_special(p.fragment) if p else 0
    )

    # URLs uniques par IP
    unique_urls = df_clean.groupby('ip')[url_col].transform('nunique')
    df_clean['nbre_unique_urls_per_ip'] = unique_urls

# ────────────────────────────────────────────────────────────
# E. VARIABLES LIÉES AU USER-AGENT  
# ────────────────────────────────────────────────────────────
print('  [5/6] Variables User-Agent...')

if 'user_agent' in df_clean.columns:
    df_clean['user_agent_length'] = df_clean['user_agent'].apply(
        lambda x: len(str(x)) if x != 'unknown' else 0
    )
    df_clean['user_agent_entropy'] = df_clean['user_agent'].apply(shannon_entropy)
    df_clean['user_agent_total_special_chars'] = df_clean['user_agent'].apply(count_special)

    # Détection bot (§3.4.6)
    _BOT_KEYWORDS = ['bot', 'crawler', 'spider', 'slurp', 'bingbot',
                     'googlebot', 'baidu', 'yandex', 'wget', 'curl', 'python']
    df_clean['is_bot'] = df_clean['user_agent'].apply(
        lambda x: 1 if any(k in str(x).lower() for k in _BOT_KEYWORDS) else 0
    )

# ────────────────────────────────────────────────────────────
# F. VARIABLES LIÉES AU REFERER  
# ────────────────────────────────────────────────────────────
print('  [6/6] Variables Referer...')

if 'referer' in df_clean.columns:
    df_clean['referer_is_present'] = df_clean['referer'].apply(
        lambda x: 0 if x in ('unknown', '', '-') else 1
    )

    parsed_ref = df_clean['referer'].apply(
        lambda x: urlparse(str(x)) if x not in ('unknown', '', '-') else None
    )

    df_clean['referer_path_depth'] = parsed_ref.apply(
        lambda p: len([s for s in p.path.split('/') if s]) if p else 0
    )
    df_clean['referer_path_struct_entropy'] = df_clean['referer'].apply(path_entropy)
    df_clean['nbre_total_referer_query_params'] = parsed_ref.apply(
        lambda p: len(parse_qs(p.query)) if p else 0
    )
    df_clean['referer_query_special_chars_count'] = parsed_ref.apply(
        lambda p: count_special(p.query) if p else 0
    )

print(f'\n  ✅ Feature Engineering terminé')
print(f'     Variables créées : {len(df_clean.columns)} colonnes au total')

  [1/6] Variables temporelles...
  [2/6] Variables IP...
  [3/6] Variables statut...
  [4/6] Variables URL...
  [5/6] Variables User-Agent...
  [6/6] Variables Referer...

  ✅ Feature Engineering terminé
     Variables créées : 41 colonnes au total


### 9 Encodage des variables categorielles 

In [35]:

# Encodage par rareté : score = 1 - fréquence_relative
# (méthode du mémoire)
# ============================================================

def rarity_encode(series: pd.Series) -> pd.Series:
    """
    Encode une variable catégorielle par son score de rareté.
    score = 1 - fréquence_relative
    → catégorie rare = score proche de 1
    → catégorie fréquente = score proche de 0
    """
    freq = series.value_counts(normalize=True)
    return series.map(lambda x: 1 - freq.get(x, 0))

# method_encoded  
if 'method' in df_clean.columns:
    df_clean['method_encoded'] = rarity_encode(df_clean['method'])

# status_category_encoded  
if 'status_code_category' in df_clean.columns:
    df_clean['status_code_category_encoded'] = rarity_encode(df_clean['status_code_category'])

# country_encoded  
# (colonne 'country' à ajouter via GeoIP si disponible,
#  sinon encodage sur 'hostname' comme proxy)
country_col = 'country' if 'country' in df_clean.columns else (
              'hostname' if 'hostname' in df_clean.columns else None)
if country_col:
    df_clean['country_encoded'] = rarity_encode(df_clean[country_col])

print('=' * 65)
print('p— ENCODAGE PAR RARETÉ')
print('=' * 65)

for col in ['method_encoded', 'status_code_category_encoded', 'country_encoded']:
    if col in df_clean.columns:
        s = df_clean[col]
        print(f'  {col:<35}  min={s.min():.4f}  max={s.max():.4f}  mean={s.mean():.4f}')

p— ENCODAGE PAR RARETÉ
  method_encoded                       min=0.1095  max=0.9961  mean=0.1958
  status_code_category_encoded         min=0.0000  max=0.0000  mean=0.0000


### 10 Transformation logarithmique

In [36]:

# Formule : log(1 + x)  pour gérer les zéros
# ============================================================

# Variables identifiées comme asymétriques dans §3.9
LOG_COLS = [
    'response_size',
    'ip_request_count',
    'ip_time_spread',
    'ip_error_rate',
    'ip_request_rate',
    'ip_interval_std',
    'nbre_unique_urls_per_ip',
    'url_path_struct_entropy',
    'url_path_depth',
    'nbre_total_url_query_params',
    'url_query_special_chars_count',
    'url_path_special_chars_count',
    'referer_path_struct_entropy',
    'referer_path_depth',
    'referer_query_special_chars_count',
    'nbre_total_referer_query_params',
    'method_encoded',
    'fragment_param_count',
    'special_char_fragment_count',
]

log_applied = []
for col in LOG_COLS:
    if col in df_clean.columns:
        num = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).clip(lower=0)
        df_clean[f'{col}_log'] = np.log1p(num)
        log_applied.append(col)

print('=' * 65)
print('— TRANSFORMATION LOGARITHMIQUE  log(1+x)')
print('=' * 65)
print(f'  Variables transformées : {len(log_applied)}')
for col in log_applied:
    print(f'    ✓ {col}  →  {col}_log')

§3.8 — TRANSFORMATION LOGARITHMIQUE  log(1+x)
  Variables transformées : 19
    ✓ response_size  →  response_size_log
    ✓ ip_request_count  →  ip_request_count_log
    ✓ ip_time_spread  →  ip_time_spread_log
    ✓ ip_error_rate  →  ip_error_rate_log
    ✓ ip_request_rate  →  ip_request_rate_log
    ✓ ip_interval_std  →  ip_interval_std_log
    ✓ nbre_unique_urls_per_ip  →  nbre_unique_urls_per_ip_log
    ✓ url_path_struct_entropy  →  url_path_struct_entropy_log
    ✓ url_path_depth  →  url_path_depth_log
    ✓ nbre_total_url_query_params  →  nbre_total_url_query_params_log
    ✓ url_query_special_chars_count  →  url_query_special_chars_count_log
    ✓ url_path_special_chars_count  →  url_path_special_chars_count_log
    ✓ referer_path_struct_entropy  →  referer_path_struct_entropy_log
    ✓ referer_path_depth  →  referer_path_depth_log
    ✓ referer_query_special_chars_count  →  referer_query_special_chars_count_log
    ✓ nbre_total_referer_query_params  →  nbre_total_referer_query_p

### 11 Standardisation

In [37]:

# Appliquée à toutes les variables numériques continues
# ============================================================

# Colonnes numériques à standardiser (sauf colonnes binaires et IDs)
EXCLUDE = {'is_bot', 'is_hierarchical', 'referer_is_present',
           'status_code', 'pid', 'tid'}

num_cols = [
    c for c in df_clean.columns
    if c not in EXCLUDE
    and c not in ('datetime',)
    and pd.api.types.is_numeric_dtype(
        pd.to_numeric(df_clean[c], errors='coerce')
    )
]

# Convertir en float
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0)

# Appliquer StandardScaler
scaler = StandardScaler()
df_clean[num_cols] = scaler.fit_transform(df_clean[num_cols])

print('=' * 65)
print(' — STANDARDISATION  (µ=0, σ=1)')
print('=' * 65)
print(f'  Colonnes standardisées : {len(num_cols)}')
for col in num_cols:
    print(f'    ✓ {col}')

 — STANDARDISATION  (µ=0, σ=1)
  Colonnes standardisées : 57
    ✓ ip
    ✓ method
    ✓ url
    ✓ protocol
    ✓ response_size
    ✓ referer
    ✓ user_agent
    ✓ source_file
    ✓ source_path
    ✓ log_type
    ✓ date_hour_sin
    ✓ date_hour_cos
    ✓ date_normalized
    ✓ ip_request_count
    ✓ ip_time_spread
    ✓ ip_request_rate
    ✓ ip_subnet_activity
    ✓ ip_error_rate
    ✓ ip_interval_std
    ✓ ip_method_diversity
    ✓ status_code_category
    ✓ url_path_depth
    ✓ url_path_struct_entropy
    ✓ nbre_total_url_query_params
    ✓ url_query_special_chars_count
    ✓ url_path_special_chars_count
    ✓ fragment_param_count
    ✓ special_char_fragment_count
    ✓ nbre_unique_urls_per_ip
    ✓ user_agent_length
    ✓ user_agent_entropy
    ✓ user_agent_total_special_chars
    ✓ referer_path_depth
    ✓ referer_path_struct_entropy
    ✓ nbre_total_referer_query_params
    ✓ referer_query_special_chars_count
    ✓ method_encoded
    ✓ status_code_category_encoded
    ✓ response_s


### CELLULE 12 — OPTIMISATION DES TYPES FINAUX


In [38]:


# Colonnes entières
for col in ['status_code', 'response_size', 'pid', 'tid']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Colonnes catégorielles
for col in ['log_type', 'method', 'level', 'protocol',
            'module', 'status_code_category']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('category')

# Colonnes texte
for col in ['ip', 'url', 'referer', 'user_agent',
            'hostname', 'service', 'message',
            'error_code', 'source_file']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('string')

print('=' * 65)
print('TYPES FINAUX')
print('=' * 65)
print(f'  {"Colonne":<40} {"dtype"}')
print(f'  {"-"*40} {"-"*15}')
for col, dtype in df_clean.dtypes.items():
    print(f'  {col:<40} {dtype}')

TYPES FINAUX
  Colonne                                  dtype
  ---------------------------------------- ---------------
  datetime                                 datetime64[us, UTC]
  ip                                       string
  method                                   category
  url                                      string
  protocol                                 category
  status_code                              int64
  response_size                            float64
  referer                                  string
  user_agent                               string
  source_file                              string
  source_path                              float64
  log_type                                 category
  date_hour_sin                            float64
  date_hour_cos                            float64
  date_normalized                          float64
  ip_request_count                         float64
  ip_time_spread                           float64
  ip

### 13 Export & Rapport final

In [39]:


df_clean.to_csv(OUTPUT_PATH, index=False)
size_mb = Path(OUTPUT_PATH).stat().st_size / (1024 * 1024)

print('=' * 65)
print('📊 RAPPORT FINAL — Pipeline terminé')
print('=' * 65)
print(f'  Fichier exporté : {OUTPUT_PATH}')
print(f'  Taille          : {size_mb:.2f} MB')
print()
print(f'  {"":<35} {"Avant":>10} {"Après":>10}')
print(f'  {"-"*35} {"-"*10} {"-"*10}')
print(f'  {"Lignes":<35} {len(df):>10,d} {len(df_clean):>10,d}')
print(f'  {"Colonnes":<35} {len(df.columns):>10,d} {len(df_clean.columns):>10,d}')
print(f'  {"NaN totaux":<35} {df.isnull().sum().sum():>10,d} {df_clean.isnull().sum().sum():>10,d}')
print()
print(f'  Distribution log_type :')
for log_type, count in df_clean['log_type'].value_counts().items():
    print(f'    {log_type:<20}: {count:>8,d}  ({count/len(df_clean)*100:5.1f}%)')
print()
print(f'  Période couverte :')
print(f'    Début : {df_clean["datetime"].min()}')
print(f'    Fin   : {df_clean["datetime"].max()}')
print('=' * 65)
print()
df_clean.head(5)

📊 RAPPORT FINAL — Pipeline terminé
  Fichier exporté : C:\Users\DELL\Desktop\projet\logs_output\Ancien_site web\Access_log\Access_log_cleaned.csv
  Taille          : 35.80 MB

                                           Avant      Après
  ----------------------------------- ---------- ----------
  Lignes                                  42,899     41,433
  Colonnes                                    12         62
  NaN totaux                                   0          0

  Distribution log_type :
    0.0                 :   41,433  (100.0%)

  Période couverte :
    Début : 2021-07-18 03:28:12+00:00
    Fin   : 2021-08-19 08:25:53+00:00



,datetime,ip,method,url,protocol,status_code,response_size,referer,user_agent,source_file,...,nbre_total_url_query_params_log,url_query_special_chars_count_log,url_path_special_chars_count_log,referer_path_struct_entropy_log,referer_path_depth_log,referer_query_special_chars_count_log,nbre_total_referer_query_params_log,method_encoded_log,fragment_param_count_log,special_char_fragment_count_log
0,2021-07-18 03:28:12+00:00,0.0,0.0,0.0,0.0,302,-0.093926,0.0,0.0,0.0,...,-0.456098,-0.506313,-0.238800,-0.085481,-0.082796,-0.0343,-0.033833,-0.350516,0.0,0.0
1,2021-07-18 03:29:49+00:00,0.0,0.0,0.0,0.0,302,0.873140,0.0,0.0,0.0,...,1.587920,1.954876,-0.886317,-0.085481,-0.082796,-0.0343,-0.033833,-0.350516,0.0,0.0
2,2021-07-18 03:30:53+00:00,0.0,0.0,0.0,0.0,302,0.373667,0.0,0.0,0.0,...,-0.456098,-0.506313,1.327558,-0.085481,-0.082796,-0.0343,-0.033833,-0.350516,0.0,0.0
3,2021-07-18 03:34:12+00:00,0.0,0.0,0.0,0.0,302,0.830632,0.0,0.0,0.0,...,1.587920,1.868695,-0.886317,-0.085481,-0.082796,-0.0343,-0.033833,-0.350516,0.0,0.0
4,2021-07-18 03:35:26+00:00,0.0,0.0,0.0,0.0,302,0.830632,0.0,0.0,0.0,...,1.587920,1.868695,-0.886317,-0.085481,-0.082796,-0.0343,-0.033833,-0.350516,0.0,0.0
